# MNIST Classifier — Training & Exploration

Этот notebook покрывает:
1. Загрузку и визуализацию данных
2. Обучение CNN-модели
3. Логирование экспериментов в MLflow
4. Анализ результатов и confusion matrix
5. Регистрацию модели в MLflow Registry

## 0. Установка зависимостей

In [ ]:
# !pip install tensorflow mlflow scikit-learn matplotlib seaborn

## 1. Импорты и конфигурация

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import mlflow
import mlflow.tensorflow
from mlflow.models.signature import infer_signature
from sklearn.metrics import classification_report, confusion_matrix

# Настройка MLflow
MLFLOW_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://localhost:5000')
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('mnist-classifier')

print(f'TensorFlow version: {tf.__version__}')
print(f'MLflow tracking URI: {MLFLOW_URI}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 2. Загрузка и исследование данных

In [ ]:
(X_train_raw, y_train), (X_test_raw, y_test) = tf.keras.datasets.mnist.load_data()

print(f'Train: {X_train_raw.shape}, Test: {X_test_raw.shape}')
print(f'Labels range: [{y_train.min()}, {y_train.max()}]')
print(f'Pixel range: [{X_train_raw.min()}, {X_train_raw.max()}]')

# Визуализация примеров
fig, axes = plt.subplots(2, 10, figsize=(20, 4))
for i in range(10):
    idx = np.where(y_train == i)[0][0]
    axes[0, i].imshow(X_train_raw[idx], cmap='gray')
    axes[0, i].set_title(f'Class {i}', fontsize=10)
    axes[0, i].axis('off')
    
    # Рандомный пример того же класса
    idx2 = np.where(y_train == i)[0][5]
    axes[1, i].imshow(X_train_raw[idx2], cmap='gray')
    axes[1, i].axis('off')

plt.suptitle('MNIST Dataset — Sample Images', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig('sample_images.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Распределение классов
unique, counts = np.unique(y_train, return_counts=True)
plt.figure(figsize=(10, 4))
plt.bar(unique, counts, color='steelblue', edgecolor='black')
plt.xlabel('Digit Class')
plt.ylabel('Count')
plt.title('Class Distribution in Training Set')
plt.xticks(range(10))
for x, c in zip(unique, counts):
    plt.text(x, c + 50, str(c), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=100)
plt.show()
print('Dataset is balanced ✅' if counts.std() < 500 else 'Dataset is imbalanced ⚠️')

## 3. Препроцессинг

In [ ]:
def preprocess(X):
    X = X.astype(np.float32) / 255.0
    return np.expand_dims(X, -1)  # (N, 28, 28) -> (N, 28, 28, 1)

X_train = preprocess(X_train_raw)
X_test  = preprocess(X_test_raw)

print(f'Processed train shape: {X_train.shape}')
print(f'Pixel range after norm: [{X_train.min():.2f}, {X_train.max():.2f}]')

## 4. Построение модели

In [ ]:
def build_cnn(conv_filters=(32, 64), dense_units=128, dropout=0.3, lr=0.001):
    inputs = tf.keras.Input(shape=(28, 28, 1), name='image')
    x = inputs
    for f in conv_filters:
        x = tf.keras.layers.Conv2D(f, (3,3), activation='relu', padding='same')(x)
        x = tf.keras.layers.MaxPooling2D((2,2))(x)
        x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(dense_units, activation='relu')(x)
    x = tf.keras.layers.Dropout(dropout)(x)
    outputs = tf.keras.layers.Dense(10, activation='softmax', name='predictions')(x)
    
    model = tf.keras.Model(inputs, outputs, name='mnist_cnn')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_cnn()
model.summary()

## 5. Обучение + MLflow

In [ ]:
CONFIG = {
    'epochs': 5,
    'batch_size': 128,
    'learning_rate': 0.001,
    'dropout_rate': 0.3,
    'conv_filters': [32, 64],
    'dense_units': 128,
}

with mlflow.start_run(run_name='notebook-training') as run:
    run_id = run.info.run_id
    print(f'MLflow Run ID: {run_id}')
    
    # Логируем параметры
    mlflow.log_params(CONFIG)
    mlflow.log_param('train_samples', len(X_train))
    
    # Логируем артефакты
    mlflow.log_artifact('sample_images.png')
    mlflow.log_artifact('class_distribution.png')
    
    model = build_cnn(
        conv_filters=CONFIG['conv_filters'],
        dense_units=CONFIG['dense_units'],
        dropout=CONFIG['dropout_rate'],
        lr=CONFIG['learning_rate']
    )
    
    # MLflow callback для логирования по эпохам
    class MLflowCB(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            mlflow.log_metrics({
                'train_loss': logs['loss'],
                'train_accuracy': logs['accuracy'],
                'val_loss': logs['val_loss'],
                'val_accuracy': logs['val_accuracy'],
            }, step=epoch)
    
    history = model.fit(
        X_train, y_train,
        epochs=CONFIG['epochs'],
        batch_size=CONFIG['batch_size'],
        validation_split=0.1,
        callbacks=[
            MLflowCB(),
            tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
        ],
        verbose=1
    )
    
    # Финальные метрики
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    mlflow.log_metrics({'test_loss': test_loss, 'test_accuracy': test_acc})
    print(f'\nTest accuracy: {test_acc:.4f}')
    
    # Сохраняем модель
    signature = infer_signature(X_test[:5], model.predict(X_test[:5]))
    mlflow.tensorflow.log_model(
        model,
        artifact_path='model',
        signature=signature,
        registered_model_name='mnist-classifier',
    )
    model.save('saved_model')
    
print(f'Done! Run: {run_id}')

## 6. Визуализация обучения

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train', marker='o')
ax1.plot(history.history['val_accuracy'], label='Val', marker='s')
ax1.set_title('Accuracy по эпохам')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'], label='Train', marker='o')
ax2.plot(history.history['val_loss'], label='Val', marker='s')
ax2.set_title('Loss по эпохам')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=100)
plt.show()

# Логируем в текущий run
with mlflow.start_run(run_id=run_id):
    mlflow.log_artifact('training_history.png')

## 7. Confusion Matrix

In [ ]:
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=-1)

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(10)]))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=range(10), yticklabels=range(10)
)
plt.title('Confusion Matrix — MNIST Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100)
plt.show()

with mlflow.start_run(run_id=run_id):
    mlflow.log_artifact('confusion_matrix.png')
    # Логируем per-class accuracy
    for cls in range(10):
        mask = y_test == cls
        cls_acc = (y_pred[mask] == y_test[mask]).mean()
        mlflow.log_metric(f'accuracy_class_{cls}', cls_acc)

print('✅ All artifacts logged to MLflow')

## 8. Регистрация модели как Production

In [ ]:
client = mlflow.tracking.MlflowClient()

# Получаем последнюю версию
versions = client.search_model_versions("name='mnist-classifier'")
latest = sorted(versions, key=lambda x: int(x.version), reverse=True)[0]
print(f'Latest version: {latest.version}, status: {latest.current_stage}')

# Продвигаем в Production (если accuracy достаточна)
if test_acc >= 0.97:
    client.transition_model_version_stage(
        name='mnist-classifier',
        version=latest.version,
        stage='Production',
        archive_existing_versions=True
    )
    print(f'✅ Model v{latest.version} promoted to Production (accuracy={test_acc:.4f})')
else:
    print(f'⚠️  Accuracy {test_acc:.4f} < 0.97 threshold. Keeping in Staging.')
    client.transition_model_version_stage(
        name='mnist-classifier',
        version=latest.version,
        stage='Staging'
    )

## 9. Тест инференса

In [ ]:
import sys
sys.path.insert(0, '..')
from src.model.predict import ModelPredictor

predictor = ModelPredictor(model_path='saved_model')

# Тест на нескольких примерах
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    idx = np.random.randint(len(X_test_raw))
    img = X_test_raw[idx]
    true_label = y_test[idx]
    
    classes, confidence = predictor.predict_class(img)
    pred = int(classes[0])
    conf = float(confidence[0])
    
    ax.imshow(img, cmap='gray')
    color = 'green' if pred == true_label else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred} ({conf:.1%})', 
                 color=color, fontsize=9)
    ax.axis('off')

plt.suptitle('Model Inference on Random Test Samples', fontsize=12)
plt.tight_layout()
plt.show()
print('🎯 Inference test complete!')